# Talks markdown generator for academicpages

Takes a TSV of talks with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `talks.py`. Run either from the `markdown_generator` folder after replacing `talks.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases, rather than Stuart's non-standard TSV format and citation style.

In [1]:
import pandas as pd
import os

## Data format

The TSV needs to have the following columns: title, type, url_slug, venue, date, location, talk_url, description, with a header at the top. Many of these fields can be blank, but the columns must be in the TSV.

- Fields that cannot be blank: `title`, `url_slug`, `date`. All else can be blank. `type` defaults to "Talk" 
- `date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. 
    - The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/talks/YYYY-MM-DD-[url_slug]`
    - The combination of `url_slug` and `date` must be unique, as it will be the basis for your filenames

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [2]:
!cat talks.tsv

title	type	url_slug	venue	date	location	talk_url	description
Ion Temperature Anisotropy Limits from Magnetic Curvature Scattering in Magnetotail Reconnection Jets	Talk	talk-37	Swedish Space Plasma Meeting	2026-03-26	Stockholm, Sweden	 https://www.kth.se/emp/sspm-2026/swedish-space-plasma-meeting-2026-1.1447149
Ion Temperature Anisotropy Limits from Magnetic Curvature Scattering in Magnetotail Reconnection Jets	Talk	talk-36	IRF-U seminars	2026-02-04	Uppsala, Sweden	 https://www.space.irfu.se/seminars/
Particle Energization in Magnetic Reconnection Jets	Invited Talk	talk-35	Subatomic, High-Energy, and Plasma physics seminars	2025-12-05	Göteborg, Sweden	
Non-Maxwellianity of ion velocity distributions in the Earth's magnetosheath	Talk	talk-35	IRF-U seminars	2025-11-12	Uppsala, Sweden	 https://www.space.irfu.se/seminars/
Non-Maxwellianity of Ion Velocity Distributions in the Earth's Magnetosheath	Talk	talk-34	Joint Cluster Plasma Observatory Workshop	2025-10-23	Paris, France	https://cluste

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [3]:
talks = pd.read_csv("talks.tsv", sep="\t", header=0)
talks

,title,type,url_slug,venue,date,location,talk_url,description
0,Ion Temperature Anisotropy Limits from Magneti...,Talk,talk-37,Swedish Space Plasma Meeting,2026-03-26,"Stockholm, Sweden",https://www.kth.se/emp/sspm-2026/swedish-spac...,NaN
1,Ion Temperature Anisotropy Limits from Magneti...,Talk,talk-36,IRF-U seminars,2026-02-04,"Uppsala, Sweden",https://www.space.irfu.se/seminars/,NaN
2,Particle Energization in Magnetic Reconnection...,Invited Talk,talk-35,"Subatomic, High-Energy, and Plasma physics sem...",2025-12-05,"Göteborg, Sweden",NaN,NaN
3,Non-Maxwellianity of ion velocity distribution...,Talk,talk-35,IRF-U seminars,2025-11-12,"Uppsala, Sweden",https://www.space.irfu.se/seminars/,NaN
4,Non-Maxwellianity of Ion Velocity Distribution...,Talk,talk-34,Joint Cluster Plasma Observatory Workshop,2025-10-23,"Paris, France",https://clusterpmo.sciencesconf.org/program/gr...,
5,Particle Energization in Magnetic Reconnection...,Invited Talk,talk-33,Workshop on kinetic physics of astrophysical p...,2025-06-20,"Paris, France",https://indico.in2p3.fr/event/36287/contributi...,
6,Electron Heating by Parallel Electric Fields i...,Invited Talk,talk-32,2nd European Conference on Magnetic Reconnecti...,2025-06-18,"Torino, Italy",https://ecmrp2.sciencesconf.org/data/pages/Lou...,
7,Electron Heating by Parallel Electric Fields i...,Talk,talk-31,11th MMS Community Workshop,2025-05-12,"Paris, France",https://10thmmsanniv.sciencesconf.org/641698,NaN
8,Particle Energization in Magnetic Reconnection...,Invited Talk,talk-30,"Department of Earth, Planetary, and Space Scie...",2025-04-04,"Los Angeles, USA",NaN,NaN
9,Particle Energization in Magnetic Reconnection...,Invited Talk,talk-29,JPP Frontiers of Plasma Physics Colloquium,2025-03-20,online,https://cassyni.com/events/FHdDJTTPT34iSmNc32QqL9,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [4]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    if type(text) is str:
        return "".join(html_escape_table.get(c,c) for c in text)
    else:
        return "False"

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [5]:
loc_dict = {}

for row, item in talks.iterrows():
    
    md_filename = str(item.date) + "-" + item.url_slug + ".md"
    html_filename = str(item.date) + "-" + item.url_slug 
    year = item.date[:4]
    
    md = "---\ntitle: \""   + item.title + '"\n'
    md += "collection: talks" + "\n"
    
    if len(str(item.type)) > 3:
        md += 'type: "' + item.type + '"\n'
    else:
        md += 'type: "Talk"\n'
    
    md += "permalink: /talks/" + html_filename + "\n"
    
    if len(str(item.venue)) > 3:
        md += 'venue: "' + item.venue + '"\n'
        
    if len(str(item.location)) > 3:
        md += "date: " + str(item.date) + "\n"
    
    if len(str(item.location)) > 3:
        md += 'location: "' + str(item.location) + '"\n'
           
    md += "---\n"
    
    
    if len(str(item.talk_url)) > 3:
        md += "\n[More information here](" + item.talk_url + ")\n" 
        
    
    if len(str(item.description)) > 3:
        md += "\n" + html_escape(item.description) + "\n"
        
        
    md_filename = os.path.basename(md_filename)
    #print(md)
    
    with open("../_talks/" + md_filename, 'w') as f:
        f.write(md)

These files are in the talks directory, one directory below where we're working from.

In [6]:
!ls ../_talks

2020-02-21-talk-1.md  2023-03-13-talk-14.md 2024-12-03-talk-27.md
2020-03-03-talk-2.md  2023-06-21-talk-15.md 2025-02-17-talk-28.md
2020-04-16-talk-3.md  2023-07-10-talk-16.md 2025-03-20-talk-29.md
2020-07-22-talk-4.md  2023-07-19-talk-17.md 2025-04-04-talk-30.md
2021-02-10-talk-5.md  2023-09-07-talk-18.md 2025-05-12-talk-31.md
2021-03-10-talk-6.md  2023-10-24-talk-19.md 2025-06-18-talk-32.md
2021-07-23-talk-7.md  2023-12-15-talk-20.md 2025-06-20-talk-33.md
2022-01-12-talk-8.md  2024-01-16-talk-21.md 2025-10-23-talk-34.md
2022-05-11-talk-10.md 2024-02-16-talk-22.md 2025-11-12-talk-35.md
2022-05-27-talk-9.md  2024-03-20-talk-23.md 2025-12-05-talk-35.md
2022-06-10-talk-11.md 2024-05-12-talk-24.md 2026-02-04-talk-36.md
2022-10-05-talk-12.md 2024-09-5-talk-25.md  2026-03-26-talk-37.md
2023-02-01-talk-13.md 2024-11-13-talk-26.md


In [7]:
!cat ../_talks/2013-03-01-tutorial-1.md

cat: ../_talks/2013-03-01-tutorial-1.md: No such file or directory
